# Домашня робота 3: Seoul Bike Sharing Demand

Варіант D. Джерело: [UCI Machine Learning Repository, dataset 560](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand).

Набір містить погодинну кількість орендованих велосипедів у Сеулі з 01.12.2017 по 30.11.2018 разом із погодою та календарними ознаками. Беру файл `SeoulBikeData.csv` повністю: 8 760 рядків (365 днів × 24 години).

План: запуск PostgreSQL через `pgserver` → завантаження CSV у staging через `COPY FROM STDIN` → типізована таблиця → аудит якості → розподіли й аномалії → десять DQL/EDA-запитів → reflection.

## 1. Setup і завантаження

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo sqlparse

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
# psycopg2 is pinned explicitly: COPY below uses its copy_expert(),
# while SQLAlchemy 2.1+ maps plain postgresql:// to psycopg 3
uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(uri, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Допоміжна функція `execute_sql()` виконує скрипт з одного чи кількох SQL-операторів в одній транзакції: для `SELECT` показує DataFrame, для DML/DDL друкує кількість змінених рядків.

In [2]:
from sqlalchemy.exc import SQLAlchemyError
from IPython.display import display
import sqlparse

def execute_sql(
    sql,
    params=None,
    return_result=False,
    output_commands=True,
):
    """
    Execute a SQL script containing one or many SQL statements.

    Every statement is executed in order using the SAME database connection.

    Output:
    - SELECT / ... RETURNING -> displays a pandas DataFrame
    - INSERT / UPDATE / DELETE -> prints affected row count
    - CREATE / DROP / ALTER, etc. -> prints success
    - If output_commands=True, prints each SQL statement before execution
    - All statements are committed together if the whole script succeeds
    - On error, the transaction is rolled back

    Parameters
    ----------
    sql : str
        One SQL statement or a script containing multiple statements.

    params : dict, optional
        Named SQLAlchemy parameters, e.g. {"id": 1} for ":id".

    return_result : bool, default=False
        If True, returns a list with execution results.

    output_commands : bool, default=True
        If True, prints each SQL statement before executing it.

    Returns
    -------
    list[dict] | None
        One result dictionary per executed SQL statement if
        return_result=True, otherwise None.
    """

    statements = [
        statement.strip()
        for statement in sqlparse.split(sql)
        if statement.strip()
    ]

    outputs = []

    try:
        with engine.begin() as conn:
            for i, statement in enumerate(statements, start=1):
                print(f"\n{'=' * 70}")
                print(f"Statement {i}/{len(statements)}")
                print(f"{'=' * 70}")

                if output_commands:
                    print(statement)
                    print("-" * 70)

                result = conn.execute(
                    text(statement),
                    params or {}
                )

                if result.returns_rows:
                    rows = result.fetchall()

                    df = pd.DataFrame(
                        rows,
                        columns=result.keys()
                    )

                    display(df)

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "result_set",
                        "data": df,
                    })

                else:
                    rowcount = result.rowcount

                    if rowcount is not None and rowcount >= 0:
                        print(
                            "Command executed successfully. "
                            f"Rows affected: {rowcount}"
                        )
                    else:
                        print("Command executed successfully.")

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "command",
                        "rowcount": rowcount,
                    })

        print(
            f"\nFinished successfully: "
            f"{len(statements)} statement(s) executed."
        )

        if return_result:
            return outputs

        return None

    except SQLAlchemyError as e:
        print(f"\nSQL error while executing statement {i}:")
        print(statement)
        print(f"\n{e}")
        raise

Файл береться з офіційного ZIP-архіву UCI. Код нижче завантажує архів і розпаковує CSV у `data/`; якщо файл там уже є (наприклад, із репозиторію), повторно він не качається.

CSV збережено в кодуванні ISO-8859-1 (символ `°` у назвах колонок), тому це кодування явно вказано і для pandas, і для `COPY`. Pandas тут потрібен лише для того, щоб глянути на перші рядки. Уся подальша обробка йде в PostgreSQL.

In [3]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / 'seoul_bike_sharing_demand.zip'
csv_path = data_dir / 'SeoulBikeData.csv'

if not csv_path.exists():
    urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extract('SeoulBikeData.csv', data_dir)

print(csv_path, f'{csv_path.stat().st_size / 1024:.0f} KB')
pd.read_csv(csv_path, encoding='ISO-8859-1', nrows=5)

data/SeoulBikeData.csv 590 KB


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0,0,0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0,0,0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0,0,0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0,0,0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0,0,0,Winter,No Holiday,Yes


## 2. Staging і DDL

### 2.1. Staging-таблиця

Усі 14 колонок у staging мають тип `TEXT` і ті самі назви, що й у CSV. Так `COPY` не впаде на порожньому рядку, `?` чи нестандартній даті, а всі перетворення будуть видні в SQL.

In [4]:
execute_sql("""
DROP TABLE IF EXISTS hw3_data_clean CASCADE;
DROP TABLE IF EXISTS hw3_data_staging CASCADE;

CREATE TABLE hw3_data_staging (
    "Date" TEXT,
    "Rented Bike Count" TEXT,
    "Hour" TEXT,
    "Temperature(°C)" TEXT,
    "Humidity(%)" TEXT,
    "Wind speed (m/s)" TEXT,
    "Visibility (10m)" TEXT,
    "Dew point temperature(°C)" TEXT,
    "Solar Radiation (MJ/m2)" TEXT,
    "Rainfall(mm)" TEXT,
    "Snowfall (cm)" TEXT,
    "Seasons" TEXT,
    "Holiday" TEXT,
    "Functioning Day" TEXT
);
""")


Statement 1/3
DROP TABLE IF EXISTS hw3_data_clean CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/3
DROP TABLE IF EXISTS hw3_data_staging CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 3/3
CREATE TABLE hw3_data_staging (
    "Date" TEXT,
    "Rented Bike Count" TEXT,
    "Hour" TEXT,
    "Temperature(°C)" TEXT,
    "Humidity(%)" TEXT,
    "Wind speed (m/s)" TEXT,
    "Visibility (10m)" TEXT,
    "Dew point temperature(°C)" TEXT,
    "Solar Radiation (MJ/m2)" TEXT,
    "Rainfall(mm)" TEXT,
    "Snowfall (cm)" TEXT,
    "Seasons" TEXT,
    "Holiday" TEXT,
    "Functioning Day" TEXT
);
----------------------------------------------------------------------
Command executed successfully.

Finished successfully: 3 statement(s) executed.


### 2.2. Завантаження через `COPY FROM STDIN`

Роздільник у файлі — кома, перший рядок — заголовок.

In [5]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='ISO-8859-1', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_data_staging (
                    "Date",
                    "Rented Bike Count",
                    "Hour",
                    "Temperature(°C)",
                    "Humidity(%)",
                    "Wind speed (m/s)",
                    "Visibility (10m)",
                    "Dew point temperature(°C)",
                    "Solar Radiation (MJ/m2)",
                    "Rainfall(mm)",
                    "Snowfall (cm)",
                    "Seasons",
                    "Holiday",
                    "Functioning Day"
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) AS loaded_rows FROM hw3_data_staging', engine)

,loaded_rows
0,8760


### 2.3. Профіль сирих значень

Перш ніж обирати типи, дивлюся на фактичні діапазони та кількість знаків після крапки в числових полях, а також на повний набір значень у категоріальних.

In [6]:
execute_sql("""
SELECT
    field_name,
    COUNT(DISTINCT raw_value) AS n_distinct,
    MIN(raw_value::NUMERIC) AS min_value,
    MAX(raw_value::NUMERIC) AS max_value,
    MAX(LENGTH(SPLIT_PART(raw_value, '.', 2))) AS max_decimals
FROM hw3_data_staging AS s
CROSS JOIN LATERAL (VALUES
    ('Rented Bike Count', s."Rented Bike Count"),
    ('Hour', s."Hour"),
    ('Temperature(°C)', s."Temperature(°C)"),
    ('Humidity(%)', s."Humidity(%)"),
    ('Wind speed (m/s)', s."Wind speed (m/s)"),
    ('Visibility (10m)', s."Visibility (10m)"),
    ('Dew point temperature(°C)', s."Dew point temperature(°C)"),
    ('Solar Radiation (MJ/m2)', s."Solar Radiation (MJ/m2)"),
    ('Rainfall(mm)', s."Rainfall(mm)"),
    ('Snowfall (cm)', s."Snowfall (cm)")
) AS f(field_name, raw_value)
GROUP BY field_name
ORDER BY field_name;

SELECT
    MIN(TO_DATE("Date", 'DD/MM/YYYY')) AS first_day,
    MAX(TO_DATE("Date", 'DD/MM/YYYY')) AS last_day,
    COUNT(DISTINCT "Date") AS n_days
FROM hw3_data_staging;

SELECT field_name, raw_value, COUNT(*) AS n_rows
FROM hw3_data_staging AS s
CROSS JOIN LATERAL (VALUES
    ('Seasons', s."Seasons"),
    ('Holiday', s."Holiday"),
    ('Functioning Day', s."Functioning Day")
) AS f(field_name, raw_value)
GROUP BY field_name, raw_value
ORDER BY field_name, raw_value;
""", output_commands=False)


Statement 1/3


,field_name,n_distinct,min_value,max_value,max_decimals
0,Dew point temperature(°C),556,-30.6,27.2,1
1,Hour,24,0,23,0
2,Humidity(%),90,0,98,0
3,Rainfall(mm),61,0,35,1
4,Rented Bike Count,2166,0,3556,0
5,Snowfall (cm),51,0,8.8,1
6,Solar Radiation (MJ/m2),345,0,3.52,2
7,Temperature(°C),546,-17.8,39.4,1
8,Visibility (10m),1789,27,2000,0
9,Wind speed (m/s),65,0,7.4,1



Statement 2/3


,first_day,last_day,n_days
0,2017-12-01,2018-11-30,365



Statement 3/3


,field_name,raw_value,n_rows
0,Functioning Day,No,295
1,Functioning Day,Yes,8465
2,Holiday,Holiday,432
3,Holiday,No Holiday,8328
4,Seasons,Autumn,2184
5,Seasons,Spring,2208
6,Seasons,Summer,2208
7,Seasons,Winter,2160



Finished successfully: 3 statement(s) executed.


**Що видно з профілю.** Усі числові поля без помилок приводяться до `NUMERIC`. Знаків після крапки не більше одного, лише в сонячній радіації два. Найширший діапазон у точки роси (від −30.6 до 27.2) і температури (від −17.8 до 39.4), тож `NUMERIC(3,1)` вистачає. `Rented Bike Count` сягає 3 556. Дати охоплюють 365 днів, з 01.12.2017 по 30.11.2018. У категоріальних полях рівно по 2–4 значення без варіацій написання. Звідси й типи в таблиці нижче.

### 2.4. Вибір типів і правила очищення

Спільне правило для всіх полів: `NULLIF(BTRIM(x), '')`, тобто пробіли по краях обрізаються, а порожній рядок стає SQL `NULL` і далі впирається в `NOT NULL`. Грошових величин у наборі немає, тож `NUMERIC(p,s)` тут використовується для вимірювань із фіксованою кількістю знаків після крапки. Для них важливо, щоб 0.1 зберігалося точно, а не як float.

| Колонка | Що означає | Формат у CSV | Тип у `hw3_data_clean` | Очищення / обмеження |
|---|---|---|---|---|
| `Id` | технічний ключ | — | `BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY` | генерується базою |
| `Date` | календарний день спостереження | `DD/MM/YYYY` | `DATE` | `TO_DATE(x, 'DD/MM/YYYY')`; часу в полі немає |
| `Hour` | година доби, початок годинного інтервалу | ціле 0–23 | `SMALLINT` | `CHECK (Hour BETWEEN 0 AND 23)` |
| `observed_at` | момент початку години за сеульським часом | виводиться з `Date` + `Hour` | `TIMESTAMPTZ` | `(Date + Hour годин) AT TIME ZONE 'Asia/Seoul'`; зручно для часових рядів і злиття з іншими джерелами |
| `Rented Bike Count` | кількість оренд за годину (target) | ціле | `INTEGER` | `CHECK (>= 0)`; максимум 3 556, тому `SMALLINT` був би впритул |
| `Temperature(°C)`, `Dew point temperature(°C)` | температура повітря і точка роси | десяткове, 1 знак, від −30.6 до 39.4 | `NUMERIC(3,1)` | до ±99.9 вистачає з запасом |
| `Humidity(%)` | відносна вологість | ціле | `SMALLINT` | `CHECK (BETWEEN 0 AND 100)` |
| `Wind speed (m/s)` | швидкість вітру | десяткове, 1 знак, 0–7.4 | `NUMERIC(3,1)` | `CHECK (>= 0)` |
| `Visibility (10m)` | видимість у десятках метрів | ціле, 27–2000 | `SMALLINT` | `CHECK (BETWEEN 0 AND 2000)`; 2000 — верхня межа приладу |
| `Solar Radiation (MJ/m2)` | сонячна радіація | десяткове, 2 знаки, 0–3.52 | `NUMERIC(4,2)` | `CHECK (>= 0)` |
| `Rainfall(mm)`, `Snowfall (cm)` | опади за годину | десяткове, 1 знак; дощ до 35, сніг до 8.8 | `NUMERIC(4,1)` | `CHECK (>= 0)` |
| `Seasons` | сезон | `Winter` / `Spring` / `Summer` / `Autumn` | `TEXT` | `CHECK (IN (...))`; бізнес-ліміту довжини немає, тому `TEXT`, а не `VARCHAR(n)` |
| `Holiday` | державне свято | `Holiday` / `No Holiday` | `BOOLEAN` | `CASE`: відоме значення → `TRUE`/`FALSE`, будь-що інше → `NULL`, і `NOT NULL` зупиняє імпорт |
| `Functioning Day` | чи працювала система прокату | `Yes` / `No` | `BOOLEAN` | те саме правило, що й для `Holiday` |

Усі поля оголошені `NOT NULL`: без дати, години чи цільової змінної рядок не має сенсу, а погодні поля в цьому наборі заповнені повністю, тому пропуск там означав би збій у джерелі. `UNIQUE ("Date", "Hour")` навмисно не ставлю в DDL, щоб у розділі 3 перевірити дублікати запитом.

In [7]:
execute_sql("""
CREATE TABLE hw3_data_clean (
    "Id" BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    "Date" DATE NOT NULL,
    "Hour" SMALLINT NOT NULL CHECK ("Hour" BETWEEN 0 AND 23),
    observed_at TIMESTAMPTZ NOT NULL,
    "Rented Bike Count" INTEGER NOT NULL CHECK ("Rented Bike Count" >= 0),
    "Temperature(°C)" NUMERIC(3, 1) NOT NULL,
    "Humidity(%)" SMALLINT NOT NULL CHECK ("Humidity(%)" BETWEEN 0 AND 100),
    "Wind speed (m/s)" NUMERIC(3, 1) NOT NULL CHECK ("Wind speed (m/s)" >= 0),
    "Visibility (10m)" SMALLINT NOT NULL CHECK ("Visibility (10m)" BETWEEN 0 AND 2000),
    "Dew point temperature(°C)" NUMERIC(3, 1) NOT NULL,
    "Solar Radiation (MJ/m2)" NUMERIC(4, 2) NOT NULL CHECK ("Solar Radiation (MJ/m2)" >= 0),
    "Rainfall(mm)" NUMERIC(4, 1) NOT NULL CHECK ("Rainfall(mm)" >= 0),
    "Snowfall (cm)" NUMERIC(4, 1) NOT NULL CHECK ("Snowfall (cm)" >= 0),
    "Seasons" TEXT NOT NULL CHECK ("Seasons" IN ('Winter', 'Spring', 'Summer', 'Autumn')),
    "Holiday" BOOLEAN NOT NULL,
    "Functioning Day" BOOLEAN NOT NULL
);
""")


Statement 1/1
CREATE TABLE hw3_data_clean (
    "Id" BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    "Date" DATE NOT NULL,
    "Hour" SMALLINT NOT NULL CHECK ("Hour" BETWEEN 0 AND 23),
    observed_at TIMESTAMPTZ NOT NULL,
    "Rented Bike Count" INTEGER NOT NULL CHECK ("Rented Bike Count" >= 0),
    "Temperature(°C)" NUMERIC(3, 1) NOT NULL,
    "Humidity(%)" SMALLINT NOT NULL CHECK ("Humidity(%)" BETWEEN 0 AND 100),
    "Wind speed (m/s)" NUMERIC(3, 1) NOT NULL CHECK ("Wind speed (m/s)" >= 0),
    "Visibility (10m)" SMALLINT NOT NULL CHECK ("Visibility (10m)" BETWEEN 0 AND 2000),
    "Dew point temperature(°C)" NUMERIC(3, 1) NOT NULL,
    "Solar Radiation (MJ/m2)" NUMERIC(4, 2) NOT NULL CHECK ("Solar Radiation (MJ/m2)" >= 0),
    "Rainfall(mm)" NUMERIC(4, 1) NOT NULL CHECK ("Rainfall(mm)" >= 0),
    "Snowfall (cm)" NUMERIC(4, 1) NOT NULL CHECK ("Snowfall (cm)" >= 0),
    "Seasons" TEXT NOT NULL CHECK ("Seasons" IN ('Winter', 'Spring', 'Summer', 'Autumn')),
    "Holiday" BOOLEAN 

### 2.5. Перенесення staging → clean через `INSERT INTO ... SELECT`

In [8]:
execute_sql("""
WITH trimmed AS (
    SELECT
        NULLIF(BTRIM("Date"), '') AS date_raw,
        NULLIF(BTRIM("Hour"), '') AS hour_raw,
        NULLIF(BTRIM("Rented Bike Count"), '') AS rentals_raw,
        NULLIF(BTRIM("Temperature(°C)"), '') AS temperature_raw,
        NULLIF(BTRIM("Humidity(%)"), '') AS humidity_raw,
        NULLIF(BTRIM("Wind speed (m/s)"), '') AS wind_raw,
        NULLIF(BTRIM("Visibility (10m)"), '') AS visibility_raw,
        NULLIF(BTRIM("Dew point temperature(°C)"), '') AS dew_point_raw,
        NULLIF(BTRIM("Solar Radiation (MJ/m2)"), '') AS solar_raw,
        NULLIF(BTRIM("Rainfall(mm)"), '') AS rainfall_raw,
        NULLIF(BTRIM("Snowfall (cm)"), '') AS snowfall_raw,
        NULLIF(BTRIM("Seasons"), '') AS seasons_raw,
        NULLIF(BTRIM("Holiday"), '') AS holiday_raw,
        NULLIF(BTRIM("Functioning Day"), '') AS functioning_raw
    FROM hw3_data_staging
),
parsed AS (
    SELECT
        TO_DATE(date_raw, 'DD/MM/YYYY') AS obs_date,
        hour_raw::SMALLINT AS obs_hour,
        rentals_raw::INTEGER AS rentals,
        temperature_raw::NUMERIC(3, 1) AS temperature,
        humidity_raw::SMALLINT AS humidity,
        wind_raw::NUMERIC(3, 1) AS wind_speed,
        visibility_raw::SMALLINT AS visibility,
        dew_point_raw::NUMERIC(3, 1) AS dew_point,
        solar_raw::NUMERIC(4, 2) AS solar_radiation,
        rainfall_raw::NUMERIC(4, 1) AS rainfall,
        snowfall_raw::NUMERIC(4, 1) AS snowfall,
        seasons_raw AS seasons,
        CASE holiday_raw
            WHEN 'Holiday' THEN TRUE
            WHEN 'No Holiday' THEN FALSE
        END AS is_holiday,
        CASE functioning_raw
            WHEN 'Yes' THEN TRUE
            WHEN 'No' THEN FALSE
        END AS is_functioning
    FROM trimmed
)
INSERT INTO hw3_data_clean (
    "Date",
    "Hour",
    observed_at,
    "Rented Bike Count",
    "Temperature(°C)",
    "Humidity(%)",
    "Wind speed (m/s)",
    "Visibility (10m)",
    "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)",
    "Rainfall(mm)",
    "Snowfall (cm)",
    "Seasons",
    "Holiday",
    "Functioning Day"
)
SELECT
    obs_date,
    obs_hour,
    (obs_date + MAKE_INTERVAL(hours => obs_hour)) AT TIME ZONE 'Asia/Seoul',
    rentals,
    temperature,
    humidity,
    wind_speed,
    visibility,
    dew_point,
    solar_radiation,
    rainfall,
    snowfall,
    seasons,
    is_holiday,
    is_functioning
FROM parsed;
""")


Statement 1/1
WITH trimmed AS (
    SELECT
        NULLIF(BTRIM("Date"), '') AS date_raw,
        NULLIF(BTRIM("Hour"), '') AS hour_raw,
        NULLIF(BTRIM("Rented Bike Count"), '') AS rentals_raw,
        NULLIF(BTRIM("Temperature(°C)"), '') AS temperature_raw,
        NULLIF(BTRIM("Humidity(%)"), '') AS humidity_raw,
        NULLIF(BTRIM("Wind speed (m/s)"), '') AS wind_raw,
        NULLIF(BTRIM("Visibility (10m)"), '') AS visibility_raw,
        NULLIF(BTRIM("Dew point temperature(°C)"), '') AS dew_point_raw,
        NULLIF(BTRIM("Solar Radiation (MJ/m2)"), '') AS solar_raw,
        NULLIF(BTRIM("Rainfall(mm)"), '') AS rainfall_raw,
        NULLIF(BTRIM("Snowfall (cm)"), '') AS snowfall_raw,
        NULLIF(BTRIM("Seasons"), '') AS seasons_raw,
        NULLIF(BTRIM("Holiday"), '') AS holiday_raw,
        NULLIF(BTRIM("Functioning Day"), '') AS functioning_raw
    FROM hw3_data_staging
),
parsed AS (
    SELECT
        TO_DATE(date_raw, 'DD/MM/YYYY') AS obs_date,
        hour_raw::

Перші рядки чистої таблиці. `TIMESTAMPTZ` зберігає абсолютний момент, а показує його в часовому поясі сесії, тому для наочності виводжу `observed_at` ще й у сеульському часі.

In [9]:
execute_sql("""
SELECT
    "Id",
    "Date",
    "Hour",
    observed_at,
    observed_at AT TIME ZONE 'Asia/Seoul' AS observed_at_seoul,
    "Rented Bike Count",
    "Temperature(°C)",
    "Seasons",
    "Holiday",
    "Functioning Day"
FROM hw3_data_clean
ORDER BY "Id"
LIMIT 5;
""", output_commands=False)


Statement 1/1


,Id,Date,Hour,observed_at,observed_at_seoul,Rented Bike Count,Temperature(°C),Seasons,Holiday,Functioning Day
0,1,2017-12-01,0,2017-11-30 15:00:00+00:00,2017-12-01 00:00:00,254,-5.2,Winter,False,True
1,2,2017-12-01,1,2017-11-30 16:00:00+00:00,2017-12-01 01:00:00,204,-5.5,Winter,False,True
2,3,2017-12-01,2,2017-11-30 17:00:00+00:00,2017-12-01 02:00:00,173,-6.0,Winter,False,True
3,4,2017-12-01,3,2017-11-30 18:00:00+00:00,2017-12-01 03:00:00,107,-6.2,Winter,False,True
4,5,2017-12-01,4,2017-11-30 19:00:00+00:00,2017-12-01 04:00:00,78,-6.0,Winter,False,True



Finished successfully: 1 statement(s) executed.


## 3. Data-quality audit

### 3.1. Кількість рядків у staging і clean

In [10]:
execute_sql("""
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_data_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_data_clean;
""", output_commands=False)


Statement 1/1


,layer,n_rows
0,staging,8760
1,clean,8760



Finished successfully: 1 statement(s) executed.


**Висновок.** У staging і clean однаково по 8 760 рядків. `INSERT ... SELECT` не має фільтра `WHERE`: жоден рядок не відкидається, а некоректний завершив би весь імпорт помилкою на `NOT NULL` чи `CHECK`. Раз вставка пройшла, кожен сирий рядок пройшов усі обмеження. 8 760 = 365 × 24, тобто файл покриває кожну годину року.

### 3.2. Перевірка дублікатів

Business key — пара `(Date, Hour)`: у файлі має бути рівно одне спостереження на кожну годину кожного дня. Окремо рахую повні дублікати за всіма змістовними колонками (без `Id` і похідного `observed_at`).

In [11]:
execute_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ("Date", "Hour")) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT ("Date", "Hour")) AS duplicate_rows
FROM hw3_data_clean;

SELECT COUNT(*) AS full_row_duplicates
FROM (
    SELECT
        "Date", "Hour", "Rented Bike Count", "Temperature(°C)", "Humidity(%)",
        "Wind speed (m/s)", "Visibility (10m)", "Dew point temperature(°C)",
        "Solar Radiation (MJ/m2)", "Rainfall(mm)", "Snowfall (cm)",
        "Seasons", "Holiday", "Functioning Day"
    FROM hw3_data_clean
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14
    HAVING COUNT(*) > 1
) AS dup;
""", output_commands=False)


Statement 1/2


,total_rows,distinct_business_keys,duplicate_rows
0,8760,8760,0



Statement 2/2


,full_row_duplicates
0,0



Finished successfully: 2 statement(s) executed.


**Висновок.** Пар `(Date, Hour)` рівно 8 760, дублікатів 0, повних копій рядків теж немає. Кожна година кожного дня представлена один раз, тому `(Date, Hour)` можна вважати природним ключем і в наступній версії схеми закріпити через `UNIQUE`. Утім, гарантії від джерела немає: перевірка ловить лише точні повтори, а, скажімо, пропущену й «домальовану» годину так не виявиш.

### 3.3. Null-rate і спеціальні маркери

Спершу null-rate для семи ключових колонок чистої таблиці.

In [12]:
execute_sql("""
SELECT
    COUNT(*) AS total_rows,
    ROUND(COUNT(*) FILTER (WHERE "Date" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_date,
    ROUND(COUNT(*) FILTER (WHERE "Rented Bike Count" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_rentals,
    ROUND(COUNT(*) FILTER (WHERE "Temperature(°C)" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_temperature,
    ROUND(COUNT(*) FILTER (WHERE "Humidity(%)" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_humidity,
    ROUND(COUNT(*) FILTER (WHERE "Seasons" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_seasons,
    ROUND(COUNT(*) FILTER (WHERE "Holiday" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_holiday,
    ROUND(COUNT(*) FILTER (WHERE "Functioning Day" IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_functioning
FROM hw3_data_clean;
""", output_commands=False)


Statement 1/1


,total_rows,pct_null_date,pct_null_rentals,pct_null_temperature,pct_null_humidity,pct_null_seasons,pct_null_holiday,pct_null_functioning
0,8760,0.00,0.00,0.00,0.00,0.00,0.00,0.00



Finished successfully: 1 statement(s) executed.


Далі — сирі значення в staging: SQL `NULL`, порожні рядки й текстові маркери пропуску (`?`, `unknown`, `null`, `n/a`, `na`, `-`) у кожній із 14 колонок.

In [13]:
execute_sql("""
SELECT
    field_name,
    COUNT(*) FILTER (WHERE raw_value IS NULL) AS sql_nulls,
    COUNT(*) FILTER (WHERE raw_value IS NOT NULL AND BTRIM(raw_value) = '') AS empty_strings,
    COUNT(*) FILTER (
        WHERE LOWER(BTRIM(raw_value)) IN ('?', 'unknown', 'null', 'n/a', 'na', '-')
    ) AS special_markers
FROM hw3_data_staging AS s
CROSS JOIN LATERAL (VALUES
    ('Date', s."Date"),
    ('Rented Bike Count', s."Rented Bike Count"),
    ('Hour', s."Hour"),
    ('Temperature(°C)', s."Temperature(°C)"),
    ('Humidity(%)', s."Humidity(%)"),
    ('Wind speed (m/s)', s."Wind speed (m/s)"),
    ('Visibility (10m)', s."Visibility (10m)"),
    ('Dew point temperature(°C)', s."Dew point temperature(°C)"),
    ('Solar Radiation (MJ/m2)', s."Solar Radiation (MJ/m2)"),
    ('Rainfall(mm)', s."Rainfall(mm)"),
    ('Snowfall (cm)', s."Snowfall (cm)"),
    ('Seasons', s."Seasons"),
    ('Holiday', s."Holiday"),
    ('Functioning Day', s."Functioning Day")
) AS f(field_name, raw_value)
GROUP BY field_name
ORDER BY field_name;
""", output_commands=False)


Statement 1/1


,field_name,sql_nulls,empty_strings,special_markers
0,Date,0,0,0
1,Dew point temperature(°C),0,0,0
2,Functioning Day,0,0,0
3,Holiday,0,0,0
4,Hour,0,0,0
5,Humidity(%),0,0,0
6,Rainfall(mm),0,0,0
7,Rented Bike Count,0,0,0
8,Seasons,0,0,0
9,Snowfall (cm),0,0,0



Finished successfully: 1 statement(s) executed.


**Висновок.** Null-rate у всіх семи колонках clean-таблиці дорівнює 0%. У staging так само немає жодного SQL `NULL`, порожнього рядка чи маркера на кшталт `?` або `unknown` в усіх 14 полях. Проте це ще не означає, що дані якісні: пропуск може ховатися за правдоподібним числом. Приклад — нульова вологість у розділі 4.3.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальних колонок

`Seasons` — основна категорія. Поруч дивлюся на `Holiday` і `Functioning Day`, бо саме в них очікую дисбаланс.

In [14]:
execute_sql("""
SELECT
    "Seasons",
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_data_clean
GROUP BY "Seasons"
ORDER BY n_rows DESC, "Seasons"
LIMIT 10;

SELECT
    "Holiday",
    "Functioning Day",
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_data_clean
GROUP BY "Holiday", "Functioning Day"
ORDER BY n_rows DESC;
""", output_commands=False)


Statement 1/2


,Seasons,n_rows,pct_rows
0,Spring,2208,25.21
1,Summer,2208,25.21
2,Autumn,2184,24.93
3,Winter,2160,24.66



Statement 2/2


,Holiday,Functioning Day,n_rows,pct_rows
0,False,True,8057,91.97
1,True,True,408,4.66
2,False,False,271,3.09
3,True,False,24,0.27



Finished successfully: 2 statement(s) executed.


**Висновок.** Сезони майже рівні (24.7–25.2%), і різниця пояснюється лише кількістю днів: у зимових місяцях 90 днів, у весняних і літніх по 92. Для моделі це не дисбаланс. Зате ознаки календаря сильно розбалансовані: святкових годин лише 4.9% (18 днів), непрацюючих 3.4% (295 годин), а комбінація «свято + прокат не працює» — всього 24 години (0.27%). По таких рідкісних групах середні ненадійні, а модель матиме замало прикладів, щоб вивчити ефект свята.

### 4.2. Базова статистика числової колонки

Target `Rented Bike Count`. До стандартного набору додаю медіану та 95-й перцентиль, щоб оцінити скошеність.

In [15]:
execute_sql("""
SELECT
    COUNT("Rented Bike Count") AS n_non_null,
    MIN("Rented Bike Count") AS min_value,
    MAX("Rented Bike Count") AS max_value,
    ROUND(AVG("Rented Bike Count")::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP("Rented Bike Count")::NUMERIC, 2) AS sd_value,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Rented Bike Count") AS median_value,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Rented Bike Count") AS p95_value
FROM hw3_data_clean;
""", output_commands=False)


Statement 1/1


,n_non_null,min_value,max_value,avg_value,sd_value,median_value,p95_value
0,8760,0,3556,704.60,645.00,504.5,2043.0



Finished successfully: 1 statement(s) executed.


**Висновок.** Розподіл оренд сильно скошений праворуч: медіана 504.5 помітно нижча за середнє 704.6, стандартне відхилення (645) майже дорівнює середньому, а максимум 3 556 майже вдвічі більший за 95-й перцентиль (2 043). Мінімум 0 — це години, коли прокат не працював (див. запит 5.10). Для регресії варто розглянути логарифм target або метрику, стійку до великих значень.

### 4.3. Outlier-check

**Статистичний (3σ)** для температури: її розподіл приблизно симетричний, тож правило тут доречне. Для `Rented Bike Count` я його не застосовую, бо розподіл сильно скошений.

In [16]:
execute_sql("""
WITH stats AS (
    SELECT
        AVG("Temperature(°C)") AS mean_value,
        STDDEV_SAMP("Temperature(°C)") AS sd_value
    FROM hw3_data_clean
    WHERE "Temperature(°C)" IS NOT NULL
)
SELECT
    t."Date",
    t."Hour",
    t."Temperature(°C)",
    ROUND(((t."Temperature(°C)" - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
FROM hw3_data_clean AS t
CROSS JOIN stats AS s
WHERE t."Temperature(°C)" IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t."Temperature(°C)" - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t."Temperature(°C)" - s.mean_value) DESC
LIMIT 20;
""", output_commands=False)


Statement 1/1


,Date,Hour,Temperature(°C),z_score



Finished successfully: 1 statement(s) executed.


**Доменний.** Відносна вологість 0% на вулиці в Сеулі фізично неправдоподібна, тож таке значення найімовірніше означає збій датчика або пропуск, записаний нулем.

In [17]:
execute_sql("""
SELECT
    "Date",
    COUNT(*) AS hours_with_zero_humidity,
    MIN("Hour") AS first_hour,
    MAX("Hour") AS last_hour,
    ROUND(AVG("Temperature(°C)"), 1) AS avg_temperature,
    ROUND(AVG("Dew point temperature(°C)"), 1) AS avg_dew_point
FROM hw3_data_clean
WHERE "Humidity(%)" = 0
GROUP BY "Date"
ORDER BY "Date";
""", output_commands=False)


Statement 1/1


,Date,hours_with_zero_humidity,first_hour,last_hour,avg_temperature,avg_dew_point
0,2018-05-19,1,7,7,11.4,4.5
1,2018-05-21,6,2,7,12.5,-2.6
2,2018-05-22,4,3,6,15.6,10.5
3,2018-05-27,2,5,7,17.6,9.2
4,2018-05-28,4,4,7,16.7,9.2



Finished successfully: 1 statement(s) executed.


**Висновок.**

- **3σ для температури** не повернуло жодного рядка: фактичний діапазон від −17.8 до 39.4 °C повністю вкладається в «середнє ± 3σ». Для клімату Сеула ці значення реалістичні, тож тут нічого не треба робити.
- **Доменна перевірка** знайшла 17 годин із вологістю 0%. Усі вони припадають на п'ять днів 19–28 травня 2018 року, переважно на ранок (2:00–7:00). При цьому середня точка роси в ці дні від −2.6 до 10.5 °C, а при нульовій вологості вона була б надзвичайно низькою. Отже, дані суперечать самі собі, і це схоже на збій датчика. Видаляти ці рядки не варто, бо оренди в них справжні. Пропоную додати прапорець `humidity_suspect`, для моделі замінити вологість на `NULL` з подальшою імпутацією і передати ці дати на перевірку власнику джерела.

## 5. Десять DQL/EDA-запитів

Кожен блок складається з бізнес-питання, SQL і висновку, написаного після виконання запиту.

### 5.1

**Business-question:** У які години робочого дня прокату середній попит найвищий?

In [18]:
execute_sql("""
SELECT
    "Hour" AS hour,
    COUNT(*) AS observations,
    ROUND(AVG("Rented Bike Count"), 2) AS avg_rentals,
    MAX("Rented Bike Count") AS max_rentals
FROM hw3_data_clean
WHERE "Functioning Day"
GROUP BY "Hour"
ORDER BY avg_rentals DESC, hour
LIMIT 5;
""", output_commands=False)


Statement 1/1


,hour,observations,avg_rentals,max_rentals
0,18,353,1554.02,3556
1,19,353,1235.78,2984
2,17,353,1177.21,2664
3,20,353,1105.30,2579
4,21,353,1066.51,2505



Finished successfully: 1 statement(s) executed.


**Interpretation:** Пік попиту припадає на вечір: о 18:00 у середньому 1 554 оренди на годину (максимум за рік 3 556), далі йдуть 19:00 (1 236), 17:00 (1 177) і 20–21:00 (понад 1 000). Ранковий пік 8:00 до п'ятірки не потрапив. Судячи з цього, прокат використовують і для дороги з роботи, і для вечірнього дозвілля. Для прогнозу година доби — одна з найсильніших ознак.

### 5.2

**Business-question:** Як відрізняється середня погодинна кількість оренд між сезонами?

In [19]:
execute_sql("""
SELECT
    "Seasons" AS season,
    COUNT(*) AS observations,
    ROUND(AVG("Rented Bike Count"), 2) AS avg_rentals,
    SUM("Rented Bike Count") AS total_rentals
FROM hw3_data_clean
WHERE "Functioning Day"
GROUP BY "Seasons"
ORDER BY avg_rentals DESC;
""", output_commands=False)


Statement 1/1


,season,observations,avg_rentals,total_rentals
0,Summer,2208,1034.07,2283234
1,Autumn,1937,924.11,1790002
2,Spring,2160,746.25,1611909
3,Winter,2160,225.54,487169



Finished successfully: 1 statement(s) executed.


**Interpretation:** Літом попит найвищий (1 034 оренди на годину), зимою найнижчий (226), тобто різниця в 4.6 раза. Осінь (924) випереджає весну (746). Порівнюю саме середні, бо кількість робочих годин у сезонах різна: восени прокат простоював 247 годин, весною 48. Сезон і температура разом задають базовий рівень попиту протягом року.

### 5.3

**Business-question:** Чи відрізняється попит у святкові та звичайні дні, коли прокат працює?

In [20]:
execute_sql("""
SELECT
    "Holiday" AS is_holiday,
    COUNT(DISTINCT "Date") AS days,
    ROUND(AVG("Rented Bike Count"), 2) AS avg_hourly_rentals,
    ROUND(SUM("Rented Bike Count") * 1.0 / COUNT(DISTINCT "Date"), 0) AS avg_daily_rentals
FROM hw3_data_clean
WHERE "Functioning Day"
GROUP BY "Holiday"
ORDER BY is_holiday;
""", output_commands=False)


Statement 1/1


,is_holiday,days,avg_hourly_rentals,avg_daily_rentals
0,False,336,739.28,17727
1,True,17,529.15,12700



Finished successfully: 1 statement(s) executed.


**Interpretation:** У свята, коли прокат працює, в середньому 529 оренд на годину і близько 12 700 на день. У звичайні дні — 739 і 17 700, тобто в свята попит приблизно на 28% нижчий. Найімовірніше, у свято зникають поїздки на роботу. Святкових днів лише 17, тому оцінка груба, і на неї можуть впливати сезони, на які ці свята випали.

### 5.4

**Business-question:** Скільки оренд припадає на ранковий пік 7–9 години у звичайні робочі дні?

In [21]:
execute_sql("""
SELECT
    "Hour" AS hour,
    COUNT(*) AS observations,
    SUM("Rented Bike Count") AS total_rentals,
    ROUND(AVG("Rented Bike Count"), 2) AS avg_rentals
FROM hw3_data_clean
WHERE "Hour" BETWEEN 7 AND 9
  AND "Functioning Day"
  AND NOT "Holiday"
GROUP BY "Hour"
ORDER BY hour;
""", output_commands=False)


Statement 1/1


,hour,observations,total_rentals,avg_rentals
0,7,336,217335,646.83
1,8,336,364458,1084.70
2,9,336,228958,681.42



Finished successfully: 1 statement(s) executed.


**Interpretation:** Ранковий пік припадає на 8:00: у середньому 1 085 оренд, у 1.6–1.7 раза більше, ніж о 7:00 (647) і о 9:00 (681). Разом за три години робочих несвяткових днів — 810 751 оренда. Такий гострий пік схожий на поїздки на роботу чи навчання. На цей час варто планувати перерозподіл велосипедів між станціями.

### 5.5

**Business-question:** Який попит у години із сильним дощем або снігопадом у несвяткові дні роботи прокату?

In [22]:
execute_sql("""
SELECT
    "Date" AS day,
    "Hour" AS hour,
    "Rainfall(mm)" AS rainfall_mm,
    "Snowfall (cm)" AS snowfall_cm,
    "Rented Bike Count" AS rentals
FROM hw3_data_clean
WHERE "Functioning Day"
  AND NOT "Holiday"
  AND ("Rainfall(mm)" >= 10 OR "Snowfall (cm)" >= 3)
ORDER BY "Rainfall(mm)" DESC, "Snowfall (cm)" DESC
LIMIT 10;
""", output_commands=False)


Statement 1/1


,day,hour,rainfall_mm,snowfall_cm,rentals
0,2018-05-16,13,35.0,0.0,151
1,2018-08-28,21,29.5,0.0,122
2,2018-07-02,20,24.0,0.0,61
3,2018-06-28,18,21.5,0.0,596
4,2018-08-29,20,21.0,0.0,19
5,2018-05-17,4,19.0,0.0,51
6,2018-08-28,18,18.5,0.0,96
7,2018-07-01,11,18.5,0.0,11
8,2018-11-08,21,18.0,0.0,21
9,2018-03-15,9,17.0,0.0,32



Finished successfully: 1 statement(s) executed.


**Interpretation:** Під час сильного дощу попит падає до кількох десятків оренд на годину: 11 при 18.5 мм опівдні 1 липня, 19 при 21 мм увечері 29 серпня. Для порівняння, звичайний вечір — понад 1 000. Виняток — 28 червня о 18:00 (596 при 21.5 мм): це година пік, і частина людей однаково їде. Усі десять рядків — про дощ, бо сортую за опадами. Годинні опади — сильна ознака, але для прогнозу наперед потрібен прогноз погоди, а не фактичне вимірювання.

### 5.6

**Business-question:** Чи є в сирому полі `Holiday` варіанти написання (інший регістр, зайві пробіли), через які перетворення в BOOLEAN могло б схибити?

In [23]:
execute_sql("""
SELECT DISTINCT
    "Holiday" AS raw_holiday,
    LENGTH("Holiday") AS raw_length,
    "Holiday" LIKE 'No %' AS starts_with_no
FROM hw3_data_staging
WHERE "Holiday" ILIKE '%holiday%'
ORDER BY raw_holiday;
""", output_commands=False)


Statement 1/1


,raw_holiday,raw_length,starts_with_no
0,Holiday,7,False
1,No Holiday,10,True



Finished successfully: 1 statement(s) executed.


**Interpretation:** У сирому полі лише два значення, `Holiday` і `No Holiday`, довжиною 7 і 10 символів, тобто без зайвих пробілів і варіацій регістру. `ILIKE` знайшов обидва, а `LIKE 'No %'` правильно відрізняє заперечення. Отже, `CASE` у перетворенні на `BOOLEAN` покриває всі реальні значення, і жоден рядок не міг отримати хибне `NULL` чи `TRUE`.

### 5.7

**Business-question:** Яку частку денних оренд у кожному сезоні дає вечірній пік 17–20 години?

In [24]:
execute_sql("""
SELECT
    "Seasons" AS season,
    SUM("Rented Bike Count") FILTER (WHERE "Hour" BETWEEN 17 AND 20) AS evening_rentals,
    SUM("Rented Bike Count") AS all_rentals,
    ROUND(
        SUM("Rented Bike Count") FILTER (WHERE "Hour" BETWEEN 17 AND 20) * 100.0
        / NULLIF(SUM("Rented Bike Count"), 0),
        2
    ) AS pct_evening
FROM hw3_data_clean
GROUP BY "Seasons"
ORDER BY pct_evening DESC;
""", output_commands=False)


Statement 1/1


,season,evening_rentals,all_rentals,pct_evening
0,Summer,676440,2283234,29.63
1,Spring,477496,1611909,29.62
2,Autumn,517062,1790002,28.89
3,Winter,119527,487169,24.54



Finished successfully: 1 statement(s) executed.


**Interpretation:** Навесні, влітку й восени на чотири вечірні години (17–20) припадає 29–30% усіх оренд сезону, хоча рівномірно було б лише 16.7%. Взимку частка нижча — 24.5%: темніє рано й холодно, тож вечірні поїздки скорочуються сильніше за денні. `NULLIF` у знаменнику страхує від ділення на нуль, якщо колись з'явиться група без жодної оренди, наприклад період повного простою.

### 5.8

**Business-question:** Чи збігається мітка `Seasons` із сезоном, який випливає з календарного місяця?

In [25]:
execute_sql("""
WITH labeled AS (
    SELECT
        "Seasons",
        CASE
            WHEN EXTRACT(MONTH FROM "Date") IN (12, 1, 2) THEN 'Winter'
            WHEN EXTRACT(MONTH FROM "Date") IN (3, 4, 5) THEN 'Spring'
            WHEN EXTRACT(MONTH FROM "Date") IN (6, 7, 8) THEN 'Summer'
            ELSE 'Autumn'
        END AS calendar_season
    FROM hw3_data_clean
)
SELECT
    COUNT(*) AS observations,
    COUNT(*) FILTER (WHERE "Seasons" IS NOT DISTINCT FROM calendar_season) AS matching_label,
    COUNT(*) FILTER (WHERE "Seasons" IS DISTINCT FROM calendar_season) AS mismatching_label
FROM labeled;
""", output_commands=False)


Statement 1/1


,observations,matching_label,mismatching_label
0,8760,8760,0



Finished successfully: 1 statement(s) executed.


**Interpretation:** Усі 8 760 міток `Seasons` збігаються з метеорологічним сезоном за місяцем (зима — грудень–лютий і так далі), розбіжностей 0. Отже, `Seasons` — повністю похідна від дати ознака, нової інформації вона не додає. У моделі вона дублює місяць, тож варто лишити щось одне, щоб не плодити корельовані ознаки.

### 5.9

**Business-question:** У якому сезоні теплі (20–30 °C) сухі години дають найвищий середній попит?

In [26]:
execute_sql("""
SELECT
    "Seasons" AS season,
    COUNT(*) AS observations,
    ROUND(AVG("Rented Bike Count"), 2) AS avg_rentals
FROM hw3_data_clean
WHERE "Temperature(°C)" BETWEEN 20 AND 30
  AND "Rainfall(mm)" = 0
  AND "Snowfall (cm)" = 0
  AND "Functioning Day"
  AND "Seasons" IN ('Spring', 'Summer', 'Autumn')
GROUP BY "Seasons"
ORDER BY avg_rentals DESC, season;
""", output_commands=False)


Statement 1/1


,season,observations,avg_rentals
0,Spring,328,1458.43
1,Autumn,450,1336.39
2,Summer,1351,1162.98



Finished successfully: 1 statement(s) executed.


**Interpretation:** За однакових умов (20–30 °C, без опадів, прокат працює) найвищий середній попит навесні (1 458), далі осінь (1 336), а літо останнє (1 163), хоча воно має найбільше таких годин (1 351). Найімовірніше, в літню вибірку потрапляє багато теплих нічних годин і задушливих днів із високою вологістю. Отже, температура впливає на попит нелінійно, і за однакових градусів відіграють роль година доби та вологість.

### 5.10

**Business-question:** Чи бувають нульові оренди в дні роботи прокату, ненульові — у дні простою, або NULL у target?

In [27]:
execute_sql("""
SELECT
    "Functioning Day" AS functioning,
    COUNT(*) AS observations,
    COUNT(*) FILTER (WHERE "Rented Bike Count" = 0) AS zero_rentals,
    COUNT(*) FILTER (WHERE "Rented Bike Count" > 0) AS positive_rentals,
    COUNT(*) FILTER (WHERE "Rented Bike Count" IS NULL) AS null_rentals
FROM hw3_data_clean
GROUP BY "Functioning Day"
ORDER BY functioning;
""", output_commands=False)


Statement 1/1


,functioning,observations,zero_rentals,positive_rentals,null_rentals
0,False,295,295,0,0
1,True,8465,0,8465,0



Finished successfully: 1 statement(s) executed.


**Interpretation:** Усі 295 годин простою мають рівно 0 оренд, а всі 8 465 робочих — більше нуля. `NULL` у target немає. Отже, нулі в target повністю визначає `Functioning Day`, а не низький попит. Для моделі попиту ці години краще прибрати з навчання або моделювати окремо, бо інакше вони штучно тягнуть прогноз униз.

### Покриття обов'язкових конструкцій

| Конструкція | Запити |
|---|---|
| явний перелік колонок у `SELECT` | 1–10 |
| `WHERE` з `AND` / `OR` / `NOT` і дужками | 5 (також `AND`/`NOT` у 4) |
| `IN` / `BETWEEN` | 4, 7, 9 |
| `LIKE` / `ILIKE` | 6 |
| `IS NULL` / `IS NOT NULL` | 10 |
| `COALESCE` / `NULLIF` | 7 (`NULLIF` у знаменнику) |
| `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` | 8 |
| `DISTINCT` | 6 |
| `ORDER BY` + `LIMIT` | 1, 5 |
| `GROUP BY` | 1, 2, 3, 4, 7, 9, 10 |

## 6. Reflection

**Якість даних.** Файл формально чистий: жодних `NULL`, порожніх рядків, маркерів `?`/`unknown` і дублікатів за `(Date, Hour)`, усі значення проходять типізацію та `CHECK`. Реальні проблеми довелося шукати доменними перевірками: 17 годин із вологістю 0% у травні 2018-го суперечать точці роси й схожі на збій датчика. Ще одна пастка — 295 годин простою з нульовими орендами: без поля `Functioning Day` їх можна прийняти за відсутність попиту. Незручно й те, що назви колонок містять `°` та пробіли, а файл збережено в ISO-8859-1.

**Features.** Як ознаки беру годину, день тижня і місяць (їх можна вивести з `Date`), температуру, вологість, вітер, видимість, опади та `Holiday`. `Seasons` не беру, бо він повністю дублює місяць. Точку роси теж ні: вона майже цілком визначається температурою і вологістю. `Id` і сирий текст дати також відкидаю, сигналу в них немає.

**Leakage.** Усі погодні поля — це вимірювання за ту саму годину. Якщо прогнозувати попит на завтра, їх на момент прогнозу ще не існує, тож треба брати прогноз погоди. `Functioning Day` теж відомий лише постфактум, якщо простій не був запланований. Сонячна радіація корелює з годиною й погодою і прямо витоку не дає, але її теж доведеться прогнозувати.

**Pipeline.** Data engineer варто запропонувати віддавати файл в UTF-8 зі snake_case-назвами, одразу додавати мітку часу з часовим поясом, писати пропуски сенсорів як `NULL` замість 0 і гарантувати унікальність `(Date, Hour)`.

**ML-задача.** Природно лягає regression, точніше forecasting погодинного попиту, тому валідацію треба ділити хронологічно, а не випадково.